# v6 overlap40 · STDL SwinV2-Small · seed42
全量微调，physical batch=4，仅使用 Train/Val，按 `val_mIoU_fg` 选模，不评估 Test。

In [ ]:
from pathlib import Path
import json, os, sys

REPO_DIR = Path('/root/autodl-tmp/projects/lunar-linear')
PROJECT_DIR = REPO_DIR / 'LTL-Net'
DATA_ROOT = Path('/root/autodl-tmp/datasets/dataset_v6_random811_overlap40')
OUTPUT_ROOT = Path('/root/autodl-tmp/outputs')
PRETRAIN_DIR = Path('/root/autodl-tmp/pretrain')
CONFIG_FILE = 'v6_overlap40_stdl_swinv2_small_full_seed42.json'
CONFIG_PATH = PROJECT_DIR / 'configs' / CONFIG_FILE

assert REPO_DIR.is_dir(), f'仓库不存在: {REPO_DIR}'
assert CONFIG_PATH.is_file(), f'配置不存在，请先更新仓库: {CONFIG_PATH}'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert config['module'] == 'stdl_swinv2_small'
assert config['freeze_stages'] == 0 and config['automatic_test_evaluation'] is False
print('配置:', CONFIG_PATH)
print('数据:', DATA_ROOT)
print('预训练目录:', PRETRAIN_DIR)
print('需要的权重:', PRETRAIN_DIR / config['pretrain_filename'])
print('输出:', OUTPUT_ROOT / f"result_{config['run_name']}")

In [ ]:
# 环境只读检查；缺包时先在终端安装，不在 Notebook 内静默改环境。
import importlib
required = ['torch', 'torchvision', 'timm', 'segmentation_models_pytorch', 'rasterio', 'matplotlib', 'tqdm', 'yaml']
missing = []
for name in required:
    try:
        importlib.import_module(name)
    except Exception as exc:
        missing.append((name, repr(exc)))
assert not missing, f'缺少运行依赖: {missing}'
print('依赖检查通过')

In [ ]:
# 完整预检（数据/权重哈希、模型身份、batch4 512前向反向）后启动训练。
import subprocess
command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'run_autodl_stdl_swin.py'),
    '--project-dir', str(PROJECT_DIR),
    '--config', str(CONFIG_PATH),
    '--data-dir', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
    '--pretrain-dir', str(PRETRAIN_DIR),
]
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
print(' '.join(command), flush=True)
subprocess.check_call(command, cwd=PROJECT_DIR, env=env)

In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
metrics = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('下载压缩包:', Path(str(result_dir) + '.zip'))